# Error analysis of the ensemble's false positives

Joins the manual failure-mode labels shipped in `DATA/outputs/fp_error_annotation.csv` (the supervised ensemble's 66 false positives) and rebuilds the failure-mode table of the paper's quantitative error analysis appendix.

In [1]:
import pandas as pd

bench = pd.read_csv("../../DATA/outputs/benchmark.csv")
oof = pd.read_csv("../../DATA/outputs/predictions/supervised_oof/ensemble_oof.csv")
for df in (bench, oof):
    df["pred_art"] = df["pred_art"].astype(str)
m = bench.merge(oof, on=["decision_id", "chunk_id", "pred_art"])

fp = m[(m.prediction == 1) & (m.gold == 0)]
len(fp)

66

In [2]:
labels = pd.read_csv("../../DATA/outputs/fp_error_annotation.csv")
labels["pred_art"] = labels["pred_art"].astype(str)
fp = fp.merge(labels, on=["decision_id", "chunk_id", "pred_art"], validate="one_to_one")
assert len(fp) == 66

tab = fp.groupby("failure_mode").agg(
    N=("gold", "size"),
    disagreed=("agreement", lambda a: int((a == 0).sum())),
    conf_mean=("proba", "mean"),
    conf_median=("proba", "median"),
)
tab["pct"] = 100 * tab.N / tab.N.sum()
tab.loc["total"] = [tab.N.sum(), tab.disagreed.sum(), fp.proba.mean(), fp.proba.median(), 100]
tab.round(2)

,N,disagreed,conf_mean,conf_median,pct
failure_mode,,,,,
other,1.0,0.0,0.71,0.71,1.52
statutory_language_not_applied,37.0,28.0,0.75,0.76,56.06
wrong_rule,28.0,17.0,0.76,0.76,42.42
total,66.0,45.0,0.75,0.76,100.00
